# 03b · Data exploration — one place to inspect the lab

**Use this notebook to see what is in a dataset before training, and what was saved after a run.** It is independent of `09_rlhf_reward_to_weights.ipynb`; that notebook is unchanged.

Start at the top. Change **`SOURCE_NAME`** in the selector cell to switch among SFT, CPT, facts, evaluation questions, manifests and saved runs. Each later section has a small set of editable controls. After switching source, rerun from the selector downward. No model weights, training, generation, API calls or file writes are needed. Optional token counting loads only the local tokenizer.

You will inspect: file/record volume, samples and raw JSON, categories, character/word lengths, fact coverage, duplicate/missing fields, split overlap, token lengths, run metrics and model answers. Counts are calculated from the files each time, not hardcoded from an earlier session.

In [ ]:
# Load the tools used to inspect local data; this cell does not load model weights.
# pathlib.Path handles file paths; json reads JSON; re matches text patterns;
# math provides arithmetic helpers; unicodedata standardizes Unicode characters.
# Counter counts repeated fact IDs, and datetime is available for timestamps.
# pandas (pd) builds/filter tables; NumPy (np) works with numeric arrays;
# Matplotlib (plt) draws charts. yaml reads the YAML configuration safely.
# IPython display and JSON render tables and expandable JSON in the notebook.
# import ... as ... gives a library a shorter name; from ... import selects names.

from pathlib import Path
import json, re, math, unicodedata
from collections import Counter
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
from IPython.display import display, JSON



In [ ]:
# Find the project from the current directory and its parents, then read settings.
# Path / 'name' joins paths. *Path.cwd().parents expands the parent sequence.
# next(...) takes the first folder containing src/lab.py; it fails if none does.
# yaml.safe_load parses config text as data. pandas display options affect only
# how tables look here; they do not change the dataset.

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/lab.py").is_file())
CFG = yaml.safe_load((ROOT / "configs/config.yaml").read_text())
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 110)



In [ ]:
# Define a reusable path resolver, then print where this notebook will read.
# def creates a function; its body runs only when resolve_path(...) is called.
# expanduser expands ~. The conditional expression chooses an absolute path
# or attaches a relative path to ROOT; resolve() normalizes the resulting path.

def resolve_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else ROOT / path).resolve()

print("Project:", ROOT)
print("This notebook reads local files; it does not run a model.")

## 1. Source names you can select

The registry below gives short names to the main files. Run files and raw markdown documents are added automatically, so a newly saved run appears when you rerun this cell. `run:08_sft_eval`, for example, means the JSON file in `runs/` with that name.

**One row has different meanings:** SFT = question–answer pair; CPT JSONL = document; eval = question; facts = canonical fact; answer run = recorded model answer. A manifest/training log may be one metadata object, not one training example. CPT text exports contain the same data as their JSONL equivalents; the explorer treats a text file as one whole text, not a guessed number of documents.

In [ ]:
# Build a name-to-file registry so later cells can switch datasets consistently.
# Each dictionary entry records a path, data kind and the meaning of one row.
# The loops add Markdown sources and saved JSON/JSONL runs found on disk.
# glob searches one folder; rglob searches nested folders; sorted fixes the order.
# path.stem removes the extension; relative_to(ROOT) makes a project-relative path.
# This registers paths, including held-out paths; it does not read their contents.

SOURCES = {
    "facts": {"path": "data/raw/facts.json", "kind": "facts", "rows_key": "facts", "unit": "canonical facts"},
    "sft_train": {"path": "data/generated/sft_train.json", "kind": "sft", "unit": "Q&A pairs"},
    "sft_val": {"path": "data/generated/sft_val.json", "kind": "sft", "unit": "Q&A pairs"},
    "cpt_train": {"path": "data/generated/cpt_session/train.jsonl", "kind": "cpt", "unit": "documents"},
    "cpt_val": {"path": "data/generated/cpt_session/validation.jsonl", "kind": "cpt", "unit": "documents"},
    "eval_dev": {"path": "evals/dev.json", "kind": "eval", "unit": "questions"},
    "eval_heldout": {"path": "evals/heldout.json", "kind": "eval", "unit": "questions"},
    "eval_all": {"path": "evals/all.json", "kind": "eval", "unit": "questions"},
    "sft_manifest": {"path": "data/generated/sft_session/manifest.json", "kind": "metadata", "unit": "metadata objects"},
    "cpt_manifest": {"path": "data/generated/cpt_session/manifest.json", "kind": "metadata", "unit": "metadata objects"},
    "sft_seed": {"path": "data/generated/sft_session/policy_refusal_seed.json", "kind": "metadata", "unit": "authoring families"},
    "cpt_train_text": {"path": "data/generated/cpt_session/train.txt", "kind": "text", "unit": "whole text files"},
}
for path in sorted((ROOT / "data/raw").glob("*.md")):
    SOURCES["raw:" + path.stem] = {"path": str(path.relative_to(ROOT)), "kind": "text", "unit": "source documents"}
for path in sorted((ROOT / "runs").rglob("*")):
    if path.is_file() and path.suffix in (".json", ".jsonl"):
        name = path.relative_to(ROOT / "runs").with_suffix("").as_posix()
        SOURCES["run:" + name] = {"path": str(path.relative_to(ROOT)), "kind": "run", "unit": "run records"}



In [ ]:
# Turn the registry into a table of available sources before selecting one.
# The list comprehension builds one dictionary per source; DataFrame makes rows.
# stat().st_size is bytes, divided by 1024 for KiB. The conditional expression
# uses None for a missing file, avoiding an attempt to read its file size.
# display(...) renders the table; existence/size checks do not load dataset rows.

catalog = pd.DataFrame([
    {"source_name": name, "kind": spec["kind"], "path": spec["path"],
     "exists": resolve_path(spec["path"]).is_file(),
     "file_KiB": round(resolve_path(spec["path"]).stat().st_size / 1024, 1) if resolve_path(spec["path"]).is_file() else None}
    for name, spec in SOURCES.items()
])
display(catalog)

## 2. A small loader for JSON, JSONL and text

JSON may be a list directly (SFT/eval), or a dictionary containing a list (`facts`, `answers`, `responses`). JSONL stores one JSON object on each line. The loader returns both the original object and a dataframe of its records. Nested lists such as `source_fact_ids` remain lists; they are not multiplied into extra training rows.

For an unfamiliar file, supply `RECORD_KEY`, such as `answers`, `validation_history` or a dotted path like `results.answers`. `RECORD_KEY="__root__"` forces the whole top-level dictionary to be treated as metadata. The original object remains available as `raw`.

In [ ]:
# Define a loader that returns the untouched JSON object, records, a table and key.
# JSONL is read line by line; enumerate(..., 1) supplies human-readable line numbers.
# try/except reports malformed JSON with its location; 'raise ... from error'
# keeps the original error as the cause. TXT/MD become one whole-text record.
# A dotted record_key walks nested dictionaries; '__root__' keeps the root object.
# isinstance distinguishes lists, dictionaries and scalars before making rows.
# pd.json_normalize flattens nested dictionary keys into columns, while list-valued
# fields stay in their cells. The returned tuple is unpacked by the selector.

def read_source(source_spec, record_key=None):
    path = resolve_path(source_spec["path"])
    if not path.is_file():
        raise FileNotFoundError(f"Source is not available yet: {path}")
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        raw = []
        for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
            if not line.strip(): continue
            try:
                raw.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSONL at {path.name}, line {line_no}") from error
    elif suffix == ".json":
        raw = json.loads(path.read_text(encoding="utf-8"))
    elif suffix in (".txt", ".md"):
        raw = [{"id": path.stem, "text": path.read_text(encoding="utf-8")}]
    else:
        raise ValueError("Choose a JSON, JSONL, TXT or MD file")

    key = record_key if record_key is not None else source_spec.get("rows_key")
    if key is None and isinstance(raw, dict):
        key = next((k for k in ("facts", "answers", "responses", "records", "examples", "data")
                    if isinstance(raw.get(k), list)), None)
    selected = raw
    if key and key != "__root__":
        for part in key.split("."):
            if not isinstance(selected, dict) or part not in selected:
                raise KeyError(f"Record path {key!r} is not present in {path.name}")
            selected = selected[part]
    if isinstance(selected, list):
        records = [row if isinstance(row, dict) else {"value": row} for row in selected]
    elif isinstance(selected, dict):
        records = [selected]
    else:
        records = [{"value": selected}]
    frame = pd.json_normalize(records) if records else pd.DataFrame()
    # Notebook10 uses question_id, while the older run files use id.
    if "id" not in frame and "question_id" in frame:
        frame["id"] = frame["question_id"]
    return raw, records, frame, key

## 3. Change this cell to explore a different source

Start with `sft_train`, then try `cpt_train`, `facts`, `eval_dev`, `run:08_sft_eval` or `run:06_cpt_session`. Selecting `eval_heldout` explicitly lets you inspect it, but keep the final split out of data-authoring and tuning decisions. `eval_all` contains dev plus held-out questions; these are overlapping views, not three separate datasets to add together.

For a new file, set `CUSTOM_PATH` and `CUSTOM_KIND`; otherwise leave them alone. `RECORD_KEY` normally stays `None` because the loader can locate the common record arrays.

In [ ]:
# Choose which registered dataset to inspect; edit these controls before running.
# None means 'no custom override', unlike an empty string or a literal filename.
# SAMPLE_N controls previews; RANDOM_SEED makes a random preview reproducible.
# Choosing held-out questions here exposes them to you; use dev for tuning.

SOURCE_NAME = "sft_train"          # CHANGE THIS to a source_name from the catalog
CUSTOM_PATH = None                # example: "runs/my_new_result.json"
CUSTOM_KIND = "run"               # sft / cpt / eval / run / facts / metadata / text
RECORD_KEY = None                 # example: "answers", "validation_history", "__root__"
SAMPLE_N = 5
RANDOM_SEED = 42



In [ ]:
# Load the selected source and make its scope explicit before analyzing it.
# dict(...) copies the selected registry entry; a custom path creates a new entry.
# Tuple unpacking assigns the loader's four returns to raw, records, df and key.
# len(df) counts rows; df.columns lists fields; f'{number:,}' adds digit separators.
# These variables are reused below, so rerun from here after changing the source.

spec = dict(SOURCES[SOURCE_NAME]) if CUSTOM_PATH is None else {
    "path": CUSTOM_PATH, "kind": CUSTOM_KIND, "unit": "custom records"}
raw, records, df, selected_key = read_source(spec, RECORD_KEY)
print("Selected:", SOURCE_NAME if CUSTOM_PATH is None else "custom file")
print("File:", resolve_path(spec["path"]))
print("Record location:", selected_key or "top level")
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print("Meaning of a row:", spec.get("unit", "records"))
print("Columns:", df.columns.tolist())

## 4. Summary of the selected dataset

This separates **how many examples** from **how many unique facts they reference**. Repeating a fact in many questions/documents does not create new knowledge. Empty source-fact lists can be legitimate for policy/refusal examples; missing `answerable` does not mean a question is unanswerable.

In [ ]:
# Define how provenance fields become a flat list of referenced fact IDs.
# Checking 'column' in frame tests column names. extend adds each list member,
# whereas append would add the whole list as one item. dropna removes missing IDs.
# Canonical fact rows use their own id; training rows use source_fact_ids/fact_id.
# Repeated IDs stay repeated so we can measure mentions as well as unique facts.

def referenced_fact_ids(frame, kind=None):
    found = []
    if "source_fact_ids" in frame:
        for ids in frame["source_fact_ids"]:
            if isinstance(ids, list): found.extend(str(v) for v in ids if v is not None)
    if "fact_id" in frame:
        found.extend(str(v) for v in frame["fact_id"].dropna())
    if kind == "facts" and "id" in frame:
        found.extend(str(v) for v in frame["id"].dropna())
    return found



In [ ]:
# Summarize example volume, provenance coverage and missing fields separately.
# set(fact_mentions) removes duplicate IDs, not duplicate training examples.
# Series.map applies a small lambda to each value; True values sum as 1.
# df.apply(..., axis=1) examines a whole row to require both question and answer.
# eq(True), eq(False) and isna separate explicit labels from missing labels.
# Series(...).to_frame() displays the summary as a one-column table.

fact_mentions = referenced_fact_ids(df, spec["kind"])
summary = {"rows": len(df), "row_unit": spec.get("unit", "records"),
           "columns": len(df.columns), "file_KiB": round(resolve_path(spec["path"]).stat().st_size/1024, 2),
           "unique_referenced_facts": len(set(fact_mentions)) if fact_mentions else None,
           "fact_references_total": len(fact_mentions) if fact_mentions else None}
for name in ("question", "answer", "text"):
    if name in df:
        present = df[name].map(lambda x: isinstance(x, str) and bool(x.strip()))
        summary[f"nonempty_{name}_rows"] = int(present.sum())
if {"question", "answer"}.issubset(df.columns):
    summary["rows_with_both_question_and_answer"] = int(df.apply(
        lambda r: all(isinstance(r[c], str) and bool(r[c].strip()) for c in ("question", "answer")), axis=1).sum())
if "answerable" in df:
    summary["answerable_true"] = int(df["answerable"].eq(True).sum())
    summary["answerable_false"] = int(df["answerable"].eq(False).sum())
    summary["answerability_missing"] = int(df["answerable"].isna().sum())
display(pd.Series(summary, name=SOURCE_NAME).to_frame())

## 5. See samples, then open one complete JSON record

`head()` shows the beginning of a file. `sample()` gives a reproducible random selection. Neither proves the full dataset is representative. The raw JSON view shows complete text and nested provenance that a compact dataframe may shorten.

In [ ]:
# Preview the first rows and a reproducible random sample of the selected table.
# The comprehension keeps only columns that actually exist; df[SHOW_COLUMNS]
# selects those columns. head(N) takes the first N rows without shuffling.
# sample(..., random_state=...) samples rows reproducibly; min caps the request
# at the available row count. A small sample cannot prove dataset quality.

SHOW_COLUMNS = [c for c in ["id", "question", "answer", "text", "kind", "topic", "model", "attribute", "value", "answerable", "status"] if c in df]
if not SHOW_COLUMNS:
    SHOW_COLUMNS = df.columns.tolist()[:8]
print("First records:")
display(df[SHOW_COLUMNS].head(SAMPLE_N))
print("Random sample:")
display(df[SHOW_COLUMNS].sample(n=min(SAMPLE_N, len(df)), random_state=RANDOM_SEED))

In [ ]:
# Inspect one original JSON record without the table's text truncation.
# List positions start at zero. The chained comparison checks that the chosen
# position is in range; 'records and ...' short-circuits when the list is empty.
# JSON(..., expanded=True) displays nested fields in an expandable notebook view.

ROW_POSITION = 0  # CHANGE THIS: zero-based position in the selected source
if records and 0 <= ROW_POSITION < len(records):
    display(JSON(records[ROW_POSITION], expanded=True))
else:
    print("No record at this position; choose a number from 0 to", len(records)-1)

## 6. Search text or filter a category

Change `SEARCH_TEXT` to something such as `battery`, `Arc 110` or `not specified`. Add a category filter, for example `FILTER_COLUMN="kind"` and `FILTER_VALUE="refusal"`. This creates `filtered_df`; it does not change `df` or write back to the file.

In [ ]:
# Set optional text and category filters without altering the source file.
# An empty SEARCH_TEXT skips text search; FILTER_COLUMN=None skips category matching.
# FILTER_VALUE can be a string or a real boolean such as False; types matter.

SEARCH_TEXT = ""          # e.g. "battery"
FILTER_COLUMN = None      # e.g. "kind", "topic", "answerable", "model_id", "condition"
FILTER_VALUE = None        # e.g. "refusal", False, "cpt", "with_docs"
DISPLAY_MATCHES = 10



In [ ]:
# Filter a copy of df, then show a bounded preview of matching rows.
# A boolean mask has one True/False per row. |= combines matches using OR across
# text columns; regex=False treats search text literally and na=False ignores gaps.
# df[mask] keeps matching rows; .eq(value) does exact category equality.
# The copy isolates this view from df, and nothing is written back to disk.

filtered_df = df.copy()
if SEARCH_TEXT:
    search_columns = [c for c in ("question", "answer", "text", "label", "value") if c in filtered_df]
    mask = pd.Series(False, index=filtered_df.index)
    for column in search_columns:
        mask |= filtered_df[column].astype("string").str.contains(SEARCH_TEXT, case=False, regex=False, na=False)
    filtered_df = filtered_df[mask]
if FILTER_COLUMN is not None:
    if FILTER_COLUMN not in filtered_df:
        raise KeyError(f"Unknown column {FILTER_COLUMN!r}; available: {df.columns.tolist()}")
    filtered_df = filtered_df[filtered_df[FILTER_COLUMN].eq(FILTER_VALUE)]
print(f"Matches: {len(filtered_df):,} / {len(df):,}")
display(filtered_df[SHOW_COLUMNS].head(DISPLAY_MATCHES))

## 7. Count categories and inspect balance

For SFT, `kind` distinguishes factual questions, comparisons, policies and refusals. For CPT, try `variation_family`; for evals, `answerable` gives the known/unknown split. For facts, try `model` or `attribute`. A missing category column is reported, not invented.

In [ ]:
# Count category balance, including rows with a missing category.
# next((...), None) chooses the first available category column or a safe default.
# map converts dictionary/list categories to stable JSON strings so they can count.
# value_counts(dropna=False) includes missing values; division turns counts into %.
# The bar chart sorts the displayed counts for readable horizontal bars;
# max/min bound its height. plt.tight_layout avoids overlapping labels.

CATEGORY_COLUMN = next((c for c in ["kind", "variation_family", "answerable", "model", "topic", "status"] if c in df), None)
# Override the line above if wanted: CATEGORY_COLUMN = "topic"
TOP_CATEGORIES = 20
if CATEGORY_COLUMN is None:
    print("No usual category field in this source. Choose another column from", df.columns.tolist())
else:
    categories = df[CATEGORY_COLUMN].map(lambda x: json.dumps(x, sort_keys=True) if isinstance(x, (dict, list)) else x)
    counts = categories.value_counts(dropna=False)
    category_summary = pd.DataFrame({"rows": counts, "percent": 100*counts/max(len(df), 1)})
    display(category_summary.head(TOP_CATEGORIES).round(2))
    if not counts.empty:
        counts.head(TOP_CATEGORIES).sort_values().plot.barh(figsize=(9, max(3, min(9, len(counts)*0.35))))
        plt.xlabel("Number of records")
        plt.title(f"{SOURCE_NAME}: {CATEGORY_COLUMN}")
        plt.tight_layout(); plt.show()

## 8. How long are the datapoints?

Character counts and whitespace-separated word counts are quick, readable measures. They are **not model token counts**. SFT questions and answers are measured separately; the combined length here excludes the system prompt, chat-template markers and EOS. The optional tokenizer section below accounts for those.

In [ ]:
# Measure text lengths without pretending that words are model tokens.
# For each available field, map marks string-valued rows; df.loc[present, column]
# selects those rows and that column by label. Pandas aligns lengths by row index.
# .str.len() counts characters; split() separates whitespace-delimited words.
# ~present flips the boolean mask to count missing/nontext rows. quantile(.95)
# is the 95th percentile, useful for spotting long examples hidden by the mean.

TEXT_COLUMNS = [c for c in ["question", "answer", "text", "prompt", "chosen", "rejected", "response"] if c in df]
# Override for an unfamiliar source, for example: TEXT_COLUMNS = ["my_text_field"]
lengths = pd.DataFrame(index=df.index)
length_summary = []
for column in TEXT_COLUMNS:
    if column not in df: raise KeyError(column)
    present = df[column].map(lambda v: isinstance(v, str))
    texts = df.loc[present, column]
    lengths[column + "_chars"] = texts.str.len()
    lengths[column + "_words"] = texts.map(lambda text: len(text.split()))
    for unit in ("chars", "words"):
        values = lengths[column + "_" + unit].dropna()
        length_summary.append({"field": column, "unit": unit, "text_rows": len(values),
                               "missing_or_nontext": int((~present).sum()), "total": values.sum(),
                               "min": values.min(), "mean": values.mean(), "median": values.median(),
                               "p95": values.quantile(.95), "max": values.max()})


In [ ]:
# Combine question and answer lengths, then display per-field summaries.
# issubset checks both needed columns before adding them element by element.
# Combined counts exclude separators, system text and chat-template markers;
# the later tokenizer section measures those model-specific details.

if {"question_words", "answer_words"}.issubset(lengths.columns):
    lengths["qa_words"] = lengths["question_words"] + lengths["answer_words"]
    lengths["qa_chars"] = lengths["question_chars"] + lengths["answer_chars"]  # separator not included
if length_summary:
    display(pd.DataFrame(length_summary).round(2))
else:
    print("No usual text columns. This source may be facts or run metadata rather than training text.")

In [ ]:
# Plot the length distribution and inspect its longest records.
# notna().any() requires at least one measured value; dropna omits missing lengths.
# nlargest returns the largest rows, and its .index gives their original labels.
# df.loc[labels, columns] retrieves those exact rows; join aligns added lengths
# by index so each displayed example keeps its own length measurement.

LENGTH_COLUMN = next((c for c in ["qa_words", "text_words", "answer_words", "question_words"] if c in lengths), None)
HISTOGRAM_BINS = 25
if LENGTH_COLUMN is not None and lengths[LENGTH_COLUMN].notna().any():
    lengths[LENGTH_COLUMN].dropna().plot.hist(bins=HISTOGRAM_BINS, figsize=(9, 3.5), edgecolor="white")
    plt.xlabel(LENGTH_COLUMN.replace("_", " "))
    plt.ylabel("Records")
    plt.title(f"Length distribution: {SOURCE_NAME}")
    plt.tight_layout(); plt.show()
    print("Longest records — edit LENGTH_COLUMN to change the measure:")
    longest = lengths[LENGTH_COLUMN].nlargest(min(5, len(df))).index
    display(df.loc[longest, SHOW_COLUMNS].join(lengths.loc[longest, [LENGTH_COLUMN]]))
else:
    print("No text lengths to plot for this source.")

## 9. Which facts are covered, and how often?

Coverage comes from provenance fields, not a new semantic fact-check. A dataset referencing all facts can still contain poor questions or wrong answers. Policy and refusal rows may correctly have no `source_fact_ids`. In SFT, this counts references in examples; in CPT it counts references in documents.

In the canonical facts, Arc 110's fast-charge value `0` means **fast charging is unsupported**. It is not a zero-minute charging time.

In [ ]:
# Compare provenance references with the canonical fact inventory.
# json.loads reads the fact dictionary; ['facts'] selects its list of fact records.
# Counter records repetitions. map looks up each canonical ID's count, then
# fillna(0) fills absent references and astype(int) makes whole-number counts.
# Set difference finds unknown IDs. Coverage means 'referenced in metadata';
# it does not establish that the example's text states the fact correctly.

fact_df = pd.DataFrame(json.loads((ROOT / "data/raw/facts.json").read_text())["facts"])
coverage_counts = Counter(referenced_fact_ids(df, spec["kind"]))
if coverage_counts:
    coverage = fact_df[["id", "model", "attribute", "value", "unit"]].copy()
    coverage["references_in_selected_source"] = coverage["id"].map(coverage_counts).fillna(0).astype(int)
    print("Unique canonical facts referenced:", int(coverage["references_in_selected_source"].gt(0).sum()), "of", len(fact_df))
    print("Fact IDs in metadata but absent from canonical facts:", sorted(set(coverage_counts)-set(fact_df["id"])))
    display(coverage.sort_values(["references_in_selected_source", "model", "attribute"]))
else:
    print("No explicit fact references in this source; absence of metadata does not establish zero factual content.")

## 10. Missing fields and repeated text

An empty answer is suspicious in SFT but normal in an evaluation question that has not been answered yet. Repeated answer text is expected when many paraphrases share the same fact. We only inspect; nothing is deleted. A repeated question can also be legitimate in a run file containing several models/conditions.

In [ ]:
# Define a comparison key for finding near-identical spellings of text.
# NFKC normalizes compatible Unicode forms; casefold handles caseless matching
# more broadly than lower(). re.sub removes runs of non-word characters.
# Returning None for nontext avoids turning missing data into the word 'nan'.
# This is a rough duplicate detector: removing punctuation can merge meanings.

def normalise_text(text):
    if not isinstance(text, str): return None
    return re.sub(r"\W+", "", unicodedata.normalize("NFKC", text).casefold())



In [ ]:
# Count blanks, missing text and repeated normalized values field by field.
# map applies normalise_text to every cell. notna() & ne('') keeps usable keys;
# & means elementwise AND for pandas masks, while ~ negates a mask.
# nunique counts distinct keys. duplicated() marks occurrences after the first,
# so the reported duplicate count is extra rows, not the number of groups.

quality_rows = []
for column in [c for c in ("question", "answer", "text", "id") if c in df]:
    nontext = ~df[column].map(lambda v: isinstance(v, str))
    blank = df[column].map(lambda v: isinstance(v, str) and not v.strip())
    norm = df[column].map(normalise_text)
    usable = norm[norm.notna() & norm.ne("")]
    quality_rows.append({"column": column, "missing_or_nontext": int(nontext.sum()),
                         "blank": int(blank.sum()), "unique_nonempty_normalized": usable.nunique(),
                         "duplicate_rows_after_first": int(usable.duplicated().sum())})
display(pd.DataFrame(quality_rows))



In [ ]:
# Show examples belonging to repeated normalized questions/text/IDs.
# duplicated(keep=False) marks every member of each duplicate group, including
# the first. df.loc[mask, columns] keeps their original rows for human review.
# No rows are deleted: repeated answers or repeated IDs across model runs may
# be legitimate, depending on what one row in the selected source represents.

DUPLICATE_COLUMN = next((c for c in ("question", "text", "id") if c in df), None)
if DUPLICATE_COLUMN:
    norm = df[DUPLICATE_COLUMN].map(normalise_text)
    mask = norm.notna() & norm.ne("") & norm.duplicated(keep=False)
    print("Rows sharing normalized", DUPLICATE_COLUMN, ":", int(mask.sum()))
    display(df.loc[mask, SHOW_COLUMNS].head(10))

## 11. Compare training and validation volumes and overlap

Change the two source names to `cpt_train` / `cpt_val` and the overlap key to `text` for CPT. For SFT, `question` checks repeated question wording, while `family_id` checks shared scenario/fact families; CPT's grouping field is `variation_family`. Some shared facts and answer text are intentional. Exact disjointness alone does not prove independence or unseen-knowledge generalization.

`eval_all` includes both evaluation splits. CPT `.txt` and `.jsonl` are alternate representations of the same corpus. Do not add overlapping views together as extra training volume.

In [ ]:
# Choose two sources and compare their volume before checking overlap.
# The dictionary comprehension loads one dataframe per source; [2] selects
# the dataframe from read_source's four-item return tuple. The loop summarizes
# fact coverage and word counts with the same definitions for both splits.
# Use training/validation or dev sources for development, keeping held-out sealed.

SPLIT_A = "sft_train"
SPLIT_B = "sft_val"
OVERLAP_KEY = "question"  # try text, family_id, id or source_fact_ids

split_frames = {name: read_source(SOURCES[name])[2] for name in (SPLIT_A, SPLIT_B)}
split_summary = []
for name, frame in split_frames.items():
    row = {"source": name, "rows": len(frame), "unit": SOURCES[name]["unit"],
           "unique_referenced_facts": len(set(referenced_fact_ids(frame, SOURCES[name]["kind"])))}
    for column in ("question", "answer", "text"):
        if column in frame:
            texts = frame[column].map(lambda value: value if isinstance(value, str) else "")
            row[column + "_words_total"] = int(texts.map(lambda value: len(value.split())).sum())
    split_summary.append(row)
display(pd.DataFrame(split_summary))



In [ ]:
# Define unique normalized values for split-overlap analysis.
# explode turns a list-valued provenance cell into separate values for this check;
# ordinary scalar values remain scalar. map applies the same normalization to both
# splits, and set removes repetitions. None signals a missing comparison column.

def overlap_values(frame, key):
    if key not in frame: return None
    values = frame[key].explode()  # also handles list-valued fact provenance
    norm = values.map(normalise_text)
    return set(norm[norm.notna() & norm.ne("")])



In [ ]:
# Find values shared by the two selected splits, then show their source rows.
# For sets, left & right means intersection. This differs from pandas' boolean &,
# though both express an AND-like operation. The nested has_shared function wraps
# a scalar in a list when needed; any(...) detects at least one shared member.
# Overlap is a diagnostic, not automatic leakage: shared facts and duplicated
# question wording have different implications for what the model is tested on.

left = overlap_values(split_frames[SPLIT_A], OVERLAP_KEY)
right = overlap_values(split_frames[SPLIT_B], OVERLAP_KEY)
if left is None or right is None:
    print(f"{OVERLAP_KEY!r} must exist in both selected sources")
else:
    shared = left & right
    print(f"Unique normalized {OVERLAP_KEY}: {SPLIT_A}={len(left):,}; {SPLIT_B}={len(right):,}; shared={len(shared):,}")
    if shared and OVERLAP_KEY in split_frames[SPLIT_A]:
        print("Sample rows from the left split with a shared value:")
        def has_shared(value):
            vals = value if isinstance(value, list) else [value]
            return any(normalise_text(v) in shared for v in vals)
        display(split_frames[SPLIT_A].loc[split_frames[SPLIT_A][OVERLAP_KEY].map(has_shared)].head(5))

## 12. Volume overview across the main datasets

This view reads the main record files and reports their units separately. It intentionally has **no grand total**: facts, Q&A examples and tokenized windows are different objects, and evaluation views can overlap.

In [ ]:
# Build a volume overview across the named sources using each source's own row unit.
# continue skips missing files after recording their absence. read_source(...)[2]
# selects the dataframe; nested comprehensions sum words across existing text fields.
# The default list includes eval_heldout: running this cell reads that file too.
# Keep it out of tuning decisions; overlapping exports must not be added as if
# they were independent datasets. This annotation does not run the cell.

OVERVIEW_SOURCES = ["facts", "sft_train", "sft_val", "cpt_train", "cpt_val", "eval_dev", "eval_heldout"]
volume_rows = []
for name in OVERVIEW_SOURCES:
    source_spec = SOURCES[name]
    path = resolve_path(source_spec["path"])
    if not path.is_file():
        volume_rows.append({"source": name, "status": "missing", "unit": source_spec["unit"]})
        continue
    frame = read_source(source_spec)[2]
    text_columns = [c for c in ("question", "answer", "text") if c in frame]
    word_count = sum(frame[c].map(lambda x: len(x.split()) if isinstance(x, str) else 0).sum() for c in text_columns)
    volume_rows.append({"source": name, "status": "present", "records": len(frame), "unit": source_spec["unit"],
                        "file_KiB": round(path.stat().st_size/1024, 1),
                        "words_in_question_answer_text": int(word_count) if text_columns else None,
                        "unique_referenced_facts": len(set(referenced_fact_ids(frame, source_spec["kind"])))} )
display(pd.DataFrame(volume_rows))

## Optional: count tokens using this lab's local tokenizer

This loads **only the tokenizer**, never model weights, and makes no generation or API calls. Leave `COUNT_TOKENS = False` to skip it. With `TOKEN_MAX_ROWS = None`, it counts every row currently in `df`; a number counts only the first N rows. If `df` has already been filtered, these are filtered-data totals. **A sample is never silently extrapolated to the full source.**

The units depend on the source:

- **SFT:** raw question/answer counts differ from training counts. Notebook 05 adds a system message, chat role markers and final EOS. Only the answer and EOS labels contribute to loss; the full sequence is capped by `training.sft.max_seq_len` (currently 512).
- **CPT:** notebook 04 tokenizes each document separately and appends one EOS. It packs the resulting stream into windows of length L (currently 512), with stride L−1 (currently 511). Adjacent windows share one context token. Padding receives loss label `-100`.
- **Other sources:** counts below describe the listed raw string fields only. They are not provider usage, chat-formatted input size, or a training-loss token count. Numeric values, lists and missing values are not coerced into text.

In [ ]:
# Set up optional token counting; it is disabled until COUNT_TOKENS=True.
# None means count all selected rows; a positive TOKEN_MAX_ROWS limits the prefix.
# pandas builds token tables and IPython.display renders them. The empty tables
# are initialized even when counting is skipped so later guarded cells are safe.
# These counts refer to the current df selection, not an extrapolated dataset.

COUNT_TOKENS = False
TOKEN_MAX_ROWS = None  # None = all current df rows; e.g. 50 = first 50 only
RAW_TOKEN_FIELDS = ["question", "answer", "text", "prompt", "response",
                    "completion", "chosen", "rejected", "system_prompt"]

import pandas as pd
from IPython.display import display

token_stats = pd.DataFrame()       # one row per selected source row
_token_summary = {}               # volume for these selected rows only
_token_window_stats = pd.DataFrame()  # CPT windows, when applicable



In [ ]:
# If enabled, load only the local Qwen tokenizer and record counting scope.
# ast will inspect Python syntax in notebook 05 without running its training cells;
# json reads that notebook, Path resolves files, and Transformers AutoTokenizer
# turns text into model-specific token IDs without loading language-model weights.
# iloc[:N] selects the first N row positions; a boolean is rejected even though
# Python treats bool as an int subclass. local_files_only forbids downloading.
# The pad token fallback uses EOS only when this tokenizer has no pad token.

if COUNT_TOKENS:
    import ast
    import json
    from pathlib import Path
    from transformers import AutoTokenizer

    if TOKEN_MAX_ROWS is not None and (
        isinstance(TOKEN_MAX_ROWS, bool) or not isinstance(TOKEN_MAX_ROWS, int)
        or TOKEN_MAX_ROWS <= 0
    ):
        raise ValueError("TOKEN_MAX_ROWS must be None or a positive integer")
    _token_rows = df if TOKEN_MAX_ROWS is None else df.iloc[:TOKEN_MAX_ROWS]
    _token_kind = spec["kind"]
    _token_model_path = Path(CFG["model"]["base_path"]).expanduser()
    if not _token_model_path.is_absolute():
        _token_model_path = ROOT / _token_model_path
    _token_tok = AutoTokenizer.from_pretrained(
        _token_model_path, local_files_only=True, trust_remote_code=False)
    if _token_tok.pad_token is None:
        _token_tok.pad_token = _token_tok.eos_token
    _token_summary = {
        "source": SOURCE_NAME,
        "kind": _token_kind,
        "rows_in_current_df": len(df),
        "rows_selected_for_counting": len(_token_rows),
        "scope": "all current df rows" if len(_token_rows) == len(df) else "first N current df rows only",
        "extrapolated": False,
    }
    print(f"Tokenizing {len(_token_rows):,} of {len(df):,} current rows from {SOURCE_NAME}.")
    print("Current row order is retained. Counts do not imply an unfiltered-file total.")
else:
    print("Token counting skipped. Set COUNT_TOKENS = True and rerun this section to enable it.")

### SFT: inspect the actual notebook 05 masking rule

For SFT data, the next cell reads **only** the `SYSTEM` assignment and `encode` function from notebook 05. It never runs the notebook's model-loading or training cells. The source of that encoder is printed so the rule remains visible.

For each row, we count the full untruncated sequence separately from the encoder's capped result. `prompt_prefix_matches` checks the assumption behind its length-based mask: the prompt must tokenize identically when followed by the answer. `answer_content_fully_retained` can remain true if only the final EOS was cut; `answer_and_eos_fully_retained` requires both. The final loss count accounts for the causal model's internal one-token shift.

In [ ]:
# Count SFT tokens using notebook 05's actual SYSTEM and encode definitions.
# This entire if block stays together so it runs only for enabled SFT counting.
# Python's ast library parses code into syntax nodes: we select just the named
# assignment/function, then compile/exec those definitions in a small namespace.
# No training cell is executed. This reflects current local encoder code, so the
# printed definitions show exactly which tokenization rules the counts follow.
# Labels of -100 are ignored by cross-entropy; labels[1:] reflects the causal
# one-token shift. Prompt, answer, EOS and truncation are measured separately.

if COUNT_TOKENS and _token_kind == "sft":
    if _token_tok.eos_token is None or _token_tok.eos_token_id is None:
        raise ValueError("Notebook 05 SFT encoding requires an EOS token")
    _nb05 = json.loads((ROOT / "notebooks/05_supervised_finetuning.ipynb").read_text())
    # Parse notebook cells as syntax; collect definitions rather than importing/running the notebook.
    _sft_nodes, _sft_sources = {}, {}
    for _nb_cell in _nb05["cells"]:
        if _nb_cell["cell_type"] != "code":
            continue
        _source = "".join(_nb_cell["source"])
        for _node in ast.parse(_source).body:
            _name = None
            if isinstance(_node, ast.FunctionDef):
                _name = _node.name
            elif isinstance(_node, ast.Assign) and len(_node.targets) == 1:
                if isinstance(_node.targets[0], ast.Name):
                    _name = _node.targets[0].id
            if _name in ("SYSTEM", "encode"):
                if _name in _sft_nodes:
                    raise ValueError(f"Multiple {_name} definitions in notebook 05; inspect before counting")
                _sft_nodes[_name] = _node
                _sft_sources[_name] = ast.get_source_segment(_source, _node)
    if set(_sft_nodes) != {"SYSTEM", "encode"}:
        raise ValueError("Could not locate notebook 05's SYSTEM and encode definitions")
    # The selected encode function receives only the config/tokenizer globals needed by its definition.
    _sft_namespace = {"CFG": CFG, "tok": _token_tok}
    for _name in ("SYSTEM", "encode"):
        _module = ast.Module(body=[_sft_nodes[_name]], type_ignores=[])
        # compile turns selected AST into code; exec installs these definitions, not the rest of notebook 05.
        exec(compile(_module, "notebook05_selected_definition", "exec"), _sft_namespace)
    print(_sft_sources["SYSTEM"])
    print(_sft_sources["encode"])

    # Now measure each valid question/answer with the same encoder and compare before/after truncation.
    _sft_limit = CFG["training"]["sft"]["max_seq_len"]
    _counted = []
    for _position, (_index, _row) in enumerate(_token_rows.iterrows()):
        _entry = {"row_position": _position, "source_index": _index, "row_id": _row.get("id")}
        _question, _answer = _row.get("question"), _row.get("answer")
        if not (isinstance(_question, str) and _question.strip()
                and isinstance(_answer, str) and _answer.strip()):
            _counted.append({**_entry, "count_status": "invalid: nonempty question and answer strings required"})
            continue
        # Render system+user+assistant-prefix markers exactly as training does; then append answer and EOS.
        _prompt = _token_tok.apply_chat_template(
            [{"role": "system", "content": _sft_namespace["SYSTEM"]},
             {"role": "user", "content": _question}],
            tokenize=False, add_generation_prompt=True)
        _prompt_ids = _token_tok(_prompt, add_special_tokens=False)["input_ids"]
        _full_ids = _token_tok(_prompt + _answer + _token_tok.eos_token,
                               add_special_tokens=False)["input_ids"]
        # Calling the extracted encoder reproduces notebook 05's truncation and -100 prompt labels.
        _encoded = _sft_namespace["encode"]({"question": _question, "answer": _answer})
        _kept_ids, _labels = _encoded["input_ids"], _encoded["labels"]
        # Token boundaries can change at concatenation, so check that prompt IDs remain a true prefix.
        _prefix_matches = _full_ids[:len(_prompt_ids)] == _prompt_ids
        _final_eos = bool(_full_ids and _full_ids[-1] == _token_tok.eos_token_id)
        # The causal loss predicts the next token: label at position 0 is never a target, hence [1:].
        _has_supervision = any(_label != -100 for _label in _labels[1:])
        _eos_retained = (_final_eos and len(_kept_ids) == len(_full_ids)
                         and bool(_labels) and _labels[-1] == _token_tok.eos_token_id)
        # Keep raw text, template, truncation, supervision and EOS counts separate for this row.
        _counted.append({
            **_entry, "count_status": "ok",
            "question_raw_tokens": len(_token_tok(_question, add_special_tokens=False)["input_ids"]),
            "answer_raw_tokens": len(_token_tok(_answer, add_special_tokens=False)["input_ids"]),
            "chat_prompt_tokens": len(_prompt_ids),
            "full_sequence_tokens_before_limit": len(_full_ids),
            "sequence_tokens_after_limit": len(_kept_ids),
            "removed_by_limit_tokens": len(_full_ids) - len(_kept_ids),
            "masked_prompt_tokens_after_limit": sum(_label == -100 for _label in _labels),
            "supervised_tokens_after_limit_including_eos": sum(_label != -100 for _label in _labels),
            "loss_targets_after_causal_shift": sum(_label != -100 for _label in _labels[1:]),
            "prompt_prefix_matches": _prefix_matches,
            "has_supervised_target": _has_supervision,
            "answer_content_fully_retained": (_prefix_matches and _final_eos
                                                and len(_kept_ids) >= len(_full_ids) - 1),
            "eos_retained": _eos_retained,
            "answer_and_eos_fully_retained": _prefix_matches and _eos_retained,
        })
    # Make a table of every selected row, retaining invalid-row reasons but excluding them from totals.
    token_stats = pd.DataFrame(_counted)
    _valid_counts = token_stats.loc[token_stats["count_status"].eq("ok")] if len(token_stats) else token_stats
    _token_summary.update({"sequence_limit": _sft_limit, "valid_rows_counted": len(_valid_counts),
                           "invalid_rows": len(token_stats) - len(_valid_counts)})
    if len(_valid_counts):
        _token_summary.update({
            "full_tokens_before_limit": int(_valid_counts["full_sequence_tokens_before_limit"].sum()),
            "sequence_tokens_after_limit": int(_valid_counts["sequence_tokens_after_limit"].sum()),
            "supervised_loss_targets": int(_valid_counts["loss_targets_after_causal_shift"].sum()),
            "rows_truncated": int(_valid_counts["removed_by_limit_tokens"].gt(0).sum()),
            "rows_without_supervised_target": int(_valid_counts["has_supervised_target"].eq(False).sum()),
            "rows_missing_final_eos": int(_valid_counts["eos_retained"].eq(False).sum()),
            "rows_with_prompt_boundary_mismatch": int(_valid_counts["prompt_prefix_matches"].eq(False).sum()),
        })
    print("SFT totals above/below include only valid rows; invalid rows remain visible in token_stats.")

### CPT: document tokens, EOS tokens, and packed training targets

The CPT calculation follows notebook 04 exactly: tokenize each `text` without automatic special tokens, append **one real EOS per document**, then pack the selected documents in their current order. For N stream tokens and windows of length L with stride L−1, the total next-token targets are **N−1**. The first token has no preceding context; overlapping boundary tokens are context in the later window and are not scored twice.

Window storage can exceed N because of that overlap and the final padded tail. Padding is never a learning target. Empty/non-string documents invalidate the packed training-volume calculation; they are reported instead of silently dropped. If this source combines training and validation splits, choose one split before counting: actual notebook 04 packs them separately.

In [ ]:
# Count CPT's document tokens and reproduce its window arithmetic without tensors.
# Each valid document contributes its text tokens plus one EOS separator.
# The concatenated stream has N tokens; length-L windows start every L-1 tokens,
# sharing one boundary token so each next-token target is counted exactly once.
# The final window may need padding, which is not a real next-token target.
# Invalid documents stop packing totals; an assert checks the total is N-1.
# The outer if keeps every step conditional on enabled CPT token counting.

if COUNT_TOKENS and _token_kind == "cpt":
    if _token_tok.eos_token_id is None:
        raise ValueError("Notebook 04 CPT preprocessing requires an EOS token")
    _counted = []
    for _position, (_index, _row) in enumerate(_token_rows.iterrows()):
        _entry = {"row_position": _position, "source_index": _index, "row_id": _row.get("id")}
        _text = _row.get("text")
        if not isinstance(_text, str) or not _text.strip():
            _counted.append({**_entry, "count_status": "invalid: nonempty text string required"})
            continue
        # add_special_tokens=False avoids implicit markers; training adds exactly one EOS per document.
        _document_ids = _token_tok(_text, add_special_tokens=False)["input_ids"]
        _counted.append({**_entry, "count_status": "ok",
                         "document_raw_tokens": len(_document_ids),
                         "document_eos_tokens_added": 1,
                         "document_training_tokens": len(_document_ids) + 1})
    # Record document-level counts first; packing totals are meaningful only if all selected texts are valid.
    token_stats = pd.DataFrame(_counted)
    _invalid = int(token_stats["count_status"].ne("ok").sum()) if len(token_stats) else 0
    _token_summary["invalid_rows"] = _invalid
    if _invalid:
        _token_summary["packing_status"] = "not calculated: fix invalid documents first"
    elif len(token_stats):
        # N is total real stream length; L is the configured stored length of each training window.
        _N = int(token_stats["document_training_tokens"].sum())
        _L = int(CFG["training"]["cpt"]["max_seq_len"])
        if _L < 2 or _N < 2:
            raise ValueError("Notebook 04 requires at least two stream tokens and max_seq_len >= 2")
        # Same start indices as notebook 04's make_windows(), without allocating tensors.
        # range stops before N-1; each window needs at least two real tokens to predict a next token.
        _windows = [{"window_index": _i, "start_token_offset": _start,
                     "real_tokens": min(_L, _N - _start),
                     "padding_tokens": _L - min(_L, _N - _start),
                     "next_token_targets": min(_L, _N - _start) - 1}
                    for _i, _start in enumerate(range(0, _N - 1, _L - 1))]
        _token_window_stats = pd.DataFrame(_windows)
        _targets = sum(_window["next_token_targets"] for _window in _windows)
        # One-token overlap must preserve every next-token target exactly once across windows.
        assert _targets == _N - 1
        _token_summary.update({
            "packing_status": "calculated for selected documents only",
            "document_raw_tokens": int(token_stats["document_raw_tokens"].sum()),
            "eos_tokens_added": len(token_stats), "stream_tokens_including_eos": _N,
            "window_length": _L, "window_stride": _L - 1, "windows": len(_windows),
            "next_token_targets": _targets,
            "padding_tokens": sum(_window["padding_tokens"] for _window in _windows),
            "stored_window_positions": len(_windows) * _L,
        })
    else:
        _token_summary["packing_status"] = "no documents selected"

### Other sources: count available raw text fields

For evaluation questions, run outputs, facts, metadata or plain text, only string fields named in `RAW_TOKEN_FIELDS` are counted. Each column stays separate; no system prompt, chat markers, answer mask, EOS or padding is invented. A run's recorded provider token usage, if present, remains a different measurement because the provider may use another tokenizer and count hidden reasoning tokens.

In [ ]:
# For other source kinds, count raw tokens only in string-valued selected fields.
# There is no implied training template, chat marker, EOS or loss mask here.
# iterrows provides each index and row; the entry dictionary retains row identity.
# None marks unavailable counts, so missing/numeric/list cells are not tokenized
# as invented text. The nested dictionary comprehension reports each field's
# measured rows and token total separately, without extrapolating.

if COUNT_TOKENS and _token_kind not in ("sft", "cpt"):
    _fields = [_field for _field in RAW_TOKEN_FIELDS if _field in _token_rows.columns]
    _counted = []
    for _position, (_index, _row) in enumerate(_token_rows.iterrows()):
        _entry = {"row_position": _position, "source_index": _index, "row_id": _row.get("id")}
        _n_fields = 0
        for _field in _fields:
            _value = _row.get(_field)
            if isinstance(_value, str):
                _entry[_field + "_raw_tokens"] = len(_token_tok(_value, add_special_tokens=False)["input_ids"])
                _n_fields += 1
            else:
                _entry[_field + "_raw_tokens"] = None  # missing/numeric/list; not the literal text "nan"
        _entry["count_status"] = "ok" if _n_fields else "no selected string fields"
        _counted.append(_entry)
    token_stats = pd.DataFrame(_counted)
    _token_summary["raw_fields_present"] = _fields
    _token_summary["field_totals"] = {
        _field: {"rows_with_string": int(token_stats[_field + "_raw_tokens"].notna().sum()),
                 "raw_tokens": (int(token_stats[_field + "_raw_tokens"].dropna().sum())
                                if token_stats[_field + "_raw_tokens"].notna().any() else None)}
        for _field in _fields if _field + "_raw_tokens" in token_stats
    }
    if not _fields:
        print("No selected raw text fields found. Add a relevant string column name to RAW_TOKEN_FIELDS if useful.")

In [ ]:
# Display the token totals with their measured scope and a few row-level examples.
# json.dumps formats the summary; default=str handles values JSON cannot encode.
# concat combines CPT head/tail windows and drop_duplicates removes overlap.
# Column-name suffix checks choose length fields. to_numeric(errors='coerce')
# marks nonnumeric values missing; describe().T puts each measure on one row.
# Counts and means therefore exclude missing values instead of claiming zero tokens.

if COUNT_TOKENS:
    print("Measured scope and totals:")
    print(json.dumps(_token_summary, indent=2, ensure_ascii=False, default=str))
    display(token_stats.head(10))
    if len(_token_window_stats):
        print("CPT first and last windows (the tail shows actual padding):")
        display(pd.concat([_token_window_stats.head(2), _token_window_stats.tail(2)]).drop_duplicates())
    _length_columns = [_column for _column in token_stats.columns
                       if _column.endswith("_tokens") or _column.endswith("_tokens_before_limit")
                       or _column.endswith("_tokens_after_limit")
                       or _column in ("supervised_tokens_after_limit_including_eos", "loss_targets_after_causal_shift")]
    if _length_columns:
        # Only measured numeric lengths; never describe missing values as zero tokens.
        _lengths = token_stats[_length_columns].apply(pd.to_numeric, errors="coerce")
        display(_lengths.describe().T[["count", "mean", "50%", "min", "max"]])
    print("token_stats keeps every selected row for further inspection; no counts were extrapolated.")

## Saved runs: metrics and provenance in one place

The following view scans JSON run files, showing recorded fields, summary metrics and training loss separately. A file without answers is not counted as an evaluation run. Filename hints are labelled as hints, not silently treated as reliable experimental settings. `training_examples_n_train` preserves the recorded example count; it is separate from CPT documents and packed windows. CPT and SFT loss use different training targets, so a lower loss does not establish a better model across these objectives.

Some old files did not save their prompt, output cap, split or adapter path. Missing means **not recorded**, not zero, false or base. A run claiming `context=none` while its saved prompt contains official documents is flagged. Different prompts, source context, token caps or question cohorts can make score changes misleading; use notebook 10 for controlled comparisons.

Generation manifests can say training was not executed **during data preparation**. Later training logs may prove you subsequently trained; the manifest is not a live project-status indicator.

In [ ]:
# Read saved run JSON files and extract comparable metadata without running models.
# rglob finds nested files; try/except records an unreadable file and continues.
# dict.get(key, fallback) tolerates optional fields. A comprehension copies only
# scalar summary values so nested histories/answer lists do not crowd the table.
# Prompt markers are evidence of supplied context; filename hints are only hints.
# The conflict flag exposes 'context: none' metadata paired with a document prompt.

RUN_FILES = sorted((ROOT / "runs").rglob("*.json"))
run_objects, run_summary_rows = {}, []
for path in RUN_FILES:
    relative = path.relative_to(ROOT / "runs").as_posix()
    try:
        obj = json.loads(path.read_text())
    except (OSError, json.JSONDecodeError) as error:
        run_summary_rows.append({"run_file": relative, "read_error": type(error).__name__})
        continue
    run_objects[relative] = obj
    if not isinstance(obj, dict):
        run_summary_rows.append({"run_file": relative, "object_type": type(obj).__name__})
        continue
    answer_rows = obj.get("answers", obj.get("responses", []))
    run_metric_summary = obj.get("summary", {})
    metrics = obj.get("metrics", {})
    prompt = obj.get("system_prompt")
    prompt_has_docs = ("=== MERIDIAN OFFICIAL DOCUMENTS ===" in prompt or "=== DOCUMENTS ===" in prompt) if isinstance(prompt, str) else None
    filename_hint = next((c for c in ("with_docs", "engineered", "naive") if c in path.stem), None)
    conflict = obj.get("context") == "none" and prompt_has_docs is True
    row = {"run_file": relative, "saved_at": obj.get("saved_at"),
           "recorded_answers": len(answer_rows) if isinstance(answer_rows, list) else None,
           "recorded_split": obj.get("split"), "recorded_context": obj.get("context"),
           "filename_condition_hint": filename_hint, "saved_prompt_has_doc_marker": prompt_has_docs,
           "context_metadata_conflict": conflict, "recorded_max_new_tokens": obj.get("max_new_tokens"),
           "recorded_adapter": obj.get("adapter_path"),
           "train_documents": obj.get("train_documents"),
           "training_examples_n_train": obj.get("n_train"),
           "train_tokens": obj.get("train_tokens"), "train_windows": obj.get("train_windows")}
    if isinstance(run_metric_summary, dict):
        row.update({k: v for k,v in run_metric_summary.items() if not isinstance(v, (dict, list))})
    elif isinstance(run_metric_summary, list):
        row["summary_rows"] = len(run_metric_summary)  # notebook10 stores multiple model/condition summaries
    if isinstance(metrics, dict):
        row.update({"training_"+k: v for k,v in metrics.items() if not isinstance(v, (dict, list))})
    run_summary_rows.append(row)


In [ ]:
# Display a consistent subset of run metadata, leaving absent fields absent.
# The comprehension keeps only preferred columns present in this mixed run table.
# This aligns saved observations for inspection; it does not re-evaluate answers,
# recalculate training or guarantee that the runs used comparable protocols.

run_summary = pd.DataFrame(run_summary_rows)
preferred = ["run_file", "recorded_answers", "recorded_split", "recorded_context", "filename_condition_hint",
             "saved_prompt_has_doc_marker", "context_metadata_conflict", "recorded_max_new_tokens",
             "accuracy_answerable", "hallucination_rate_answerable", "over_refusal_rate", "appropriate_refusal_rate",
             "training_train_loss", "training_epoch", "training_examples_n_train", "train_documents", "train_tokens", "train_windows"]
display(run_summary[[c for c in preferred if c in run_summary]])
print("All available run-summary columns:", run_summary.columns.tolist())

## Open one run: metadata, answers and any saved training curve

Change `RUN_FILE` to any relative filename shown above. `06_cpt_session.json` has training metrics and validation-loss history; `08_sft_eval.json` has model answers and evaluation scores. A final training-loss scalar is not a per-step loss curve, and no missing history is reconstructed.

In [ ]:
# Open a chosen saved run, show metadata and plot recorded training loss if present.
# The dictionary comprehension omits bulky answer/prompt fields only from the
# preview; the complete prompt is printed separately when stored. json_normalize
# turns answer/history dictionaries into tables. get(..., fallback) supports
# several saved-run schemas. A chart is drawn only if step/epoch and loss exist;
# missing history stays missing rather than being reconstructed or fabricated.

RUN_FILE = "06_cpt_session.json"  # e.g. "08_sft_eval.json" or "04_diagnosis.json"
RUN_SAMPLES = 5
run_path = ROOT / "runs" / RUN_FILE
if not run_path.is_file():
    print("Run not found:", run_path)
else:
    run_raw = json.loads(run_path.read_text())
    if isinstance(run_raw, dict):
        # Exclude the bulky answer array and full prompt from the metadata preview.
        preview = {k:v for k,v in run_raw.items() if k not in ("answers", "responses", "historical_baseline_responses", "system_prompt")}
        display(JSON(preview, expanded=True))
        if "system_prompt" in run_raw:
            print("Saved system prompt (complete):\n", run_raw["system_prompt"])
        answers = run_raw.get("answers", run_raw.get("responses", []))
        if isinstance(answers, list) and answers:
            display(pd.json_normalize(answers).head(RUN_SAMPLES))
            print("Choose this run in SOURCE_NAME above for full sampling, filtering and length analysis.")
        history = run_raw.get("validation_history", run_raw.get("log_history", []))
        if isinstance(history, list) and history:
            history_df = pd.json_normalize(history)
            display(history_df)
            x = next((c for c in ("step", "epoch") if c in history_df), None)
            ys = [c for c in ("loss", "eval_loss") if c in history_df]
            if x and ys:
                ax = history_df.plot(x=x, y=ys, marker="o", figsize=(8, 3.5))
                ax.set_ylabel("Recorded loss"); ax.set_title(RUN_FILE)
                plt.tight_layout(); plt.show()
        else:
            print("No per-step/epoch history was stored in this file.")
    else:
        display(JSON(run_raw, expanded=False))

## Compare two saved answer sets, question by question

This is a raw-answer comparison, not a new model evaluation. Edit the two sources; for notebook10 cache/report files, filter to one `model_id` and one `condition` on each side. Duplicate question IDs are rejected to avoid multiplying rows in a many-to-many join.

The joined table shows which questions appear on both sides, whether their text matches, and both answers. Current dev ground truth is attached only when ID and question text match. Expected values omitted in an older run can then be inspected without pretending they were originally saved. Questions unique to either run remain visible in the outer join. No automatic ranking or causal conclusion is made from this view.

In [ ]:
# Choose two saved answer runs and optional filters for a fair row-by-row view.
# Filter dictionaries select a single model/condition if a run contains several.
# COMPARE_N limits the displayed comparison, not the number of rows aligned.
# The run names refer to the registry built earlier, not live model requests.

LEFT_RUN = "run:01_stock_naive"
RIGHT_RUN = "run:06_cpt_session_eval_naive"
LEFT_FILTERS = {}    # example: {"model_id": "base", "condition": "naive"}
RIGHT_FILTERS = {}   # example: {"model_id": "cpt", "condition": "naive"}
COMPARE_N = 10



In [ ]:
# Define a loader that requires one saved answer per question ID.
# Successive boolean filters keep rows whose column equals the requested value.
# issubset checks required columns; isna and duplicated reject ambiguous IDs.
# The returned columns preserve question text, answer and any existing labels.
# These guards prevent a join from silently multiplying repeated question IDs.

def answer_frame(source_name, filters):
    frame = read_source(SOURCES[source_name])[2]
    for column, value in filters.items():
        if column not in frame: raise KeyError(f"{column} missing in {source_name}")
        frame = frame[frame[column].eq(value)]
    if not {"id", "question", "answer"}.issubset(frame):
        raise ValueError(f"{source_name} is not a saved answer table with id, question and answer")
    if frame["id"].isna().any() or frame["id"].duplicated().any():
        raise ValueError(f"{source_name} has missing/repeated question IDs; filter to one model, condition and attempt first")
    return frame[[c for c in ("id", "question", "answer", "status", "answerable", "expected") if c in frame]]



In [ ]:
# Align two saved runs on question ID and expose missing/mismatched rows.
# merge(how='outer') keeps both sides; suffixes label matching column names.
# validate='one_to_one' rejects duplicate join keys, and indicator adds _merge.
# apply(..., axis=1) checks question text before attaching current dev expectations;
# matching an ID alone is insufficient evidence that the question stayed the same.
# The styled preview wraps long text; exceptions print a useful selection error.

if LEFT_RUN not in SOURCES or RIGHT_RUN not in SOURCES:
    print("Choose two existing run: names from the catalog before comparing.")
else:
    try:
        left_answers = answer_frame(LEFT_RUN, LEFT_FILTERS)
        right_answers = answer_frame(RIGHT_RUN, RIGHT_FILTERS)
        # Outer joining exposes answers missing from either side instead of silently dropping them.
        answer_comparison = left_answers.merge(right_answers, on="id", how="outer", suffixes=("_left", "_right"), indicator=True, validate="one_to_one")
        answer_comparison["same_question_text"] = answer_comparison.apply(
            lambda r: isinstance(r.get("question_left"), str) and isinstance(r.get("question_right"), str)
            and r["question_left"] == r["question_right"], axis=1)
        # Read only current dev expectations; attach one only after ID and question text agree.
        dev_by_id = {q["id"]: q for q in json.loads((ROOT / "evals/dev.json").read_text())}
        def current_expected(row):
            q = dev_by_id.get(row["id"])
            if q and row["same_question_text"] and q["question"] == row["question_left"]:
                return q["expected_value"] if q["answerable"] else "UNANSWERABLE in dev"
            return None
        answer_comparison["current_dev_expected_if_matched"] = answer_comparison.apply(current_expected, axis=1)
        print("Row alignment:"); display(answer_comparison["_merge"].value_counts())
        print("Questions with matching text:", int(answer_comparison["same_question_text"].sum()))
        cols = ["id", "same_question_text", "question_left", "question_right", "current_dev_expected_if_matched", "answer_left", "answer_right"]
        display(answer_comparison[cols].head(COMPARE_N).style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"}))
    except (ValueError, KeyError, FileNotFoundError) as error:
        print("Comparison needs a different source/filter:", error)

## Your next edits

- Switch `SOURCE_NAME` between SFT, CPT, facts, eval and a saved run.
- Change `SEARCH_TEXT`, `FILTER_COLUMN`, `CATEGORY_COLUMN` or `LENGTH_COLUMN` to investigate a question.
- Use `ROW_POSITION` to read a complete record and its provenance.
- Change the split selectors to compare training/validation size and overlap.
- Enable local token counting to distinguish raw text volume from actual training inputs and supervised tokens.
- Open a saved run and compare specific answers before interpreting an aggregate score.

The notebook leaves all source files unchanged. Record your observations separately; no training conclusion is prefilled here.